# Genomic Needle-in-a-Haystack (gNIAH) Protocol

The gNIAH protocol (Section 5.10) provides a biologically interpretable way to probe context sensitivity. A known regulatory motif is inserted into a neutral background at varying distances from the prediction center, and we measure how much the embedding changes:

$$\text{gNIAH}(d, m) = \mathbb{E}[d_Z(f(X_{\text{neutral}}), f(X_{\text{neutral}}^{+m@d}))]$$

This notebook demonstrates the protocol using synthetic models and three built-in motifs.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from ecl.models.base import SyntheticModel
from ecl.gniah import gniah_sensitivity, MOTIFS, generate_neutral_background, encode_motif

## Setup: Models and Motifs

We create two synthetic models with different decay lengths and test three regulatory motifs (CTCF, GATA, SP1).

In [ ]:
SEQ_LENGTH = 2000
rng = np.random.default_rng(42)

models = {
    "Short context (decay=50)": SyntheticModel(SEQ_LENGTH, embed_dim=64, decay_length=50.0),
    "Long context (decay=300)": SyntheticModel(SEQ_LENGTH, embed_dim=64, decay_length=300.0),
}

motif_names = ["CTCF", "GATA", "SP1"]
test_distances = np.arange(0, 800, 25)

print("Built-in motifs:")
for name, seq in MOTIFS.items():
    print(f"  {name}: {seq} ({len(seq)} bp)")

## Run gNIAH Sensitivity Analysis

In [ ]:
results = {}
for model_name, model in models.items():
    results[model_name] = {}
    for motif in motif_names:
        print(f"  {model_name} / {motif}...")
        sensitivity = gniah_sensitivity(
            model_fn=model,
            motif_name=motif,
            distances=test_distances,
            seq_length=SEQ_LENGTH,
            n_samples=15,
            rng=rng,
            show_progress=False,
        )
        results[model_name][motif] = sensitivity

print("Done.")

## Visualize gNIAH Profiles

Each panel shows one motif. Lines show how the embedding sensitivity decays with distance for each model. A model with longer effective context maintains sensitivity at greater distances.

In [ ]:
colors = {"Short context (decay=50)": "#e41a1c", "Long context (decay=300)": "#377eb8"}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for idx, motif in enumerate(motif_names):
    ax = axes[idx]
    for model_name in models:
        sens = results[model_name][motif]
        # Normalize to peak sensitivity
        peak = sens.max() if sens.max() > 0 else 1.0
        ax.plot(test_distances, sens / peak, color=colors[model_name],
                linewidth=1.5, label=model_name)
    ax.set_xlabel("Insertion distance (bp)")
    if idx == 0:
        ax.set_ylabel("Normalized gNIAH sensitivity")
    ax.set_title(f"{motif} motif ({MOTIFS[motif]})")
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=8)

plt.suptitle("gNIAH Sensitivity vs Insertion Distance", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Interpretation

- The **short-context model** shows gNIAH sensitivity that drops sharply within a few hundred bp, confirming its limited effective context.
- The **long-context model** maintains sensitivity at much greater distances, consistent with its 300 bp decay length.
- All three motifs (CTCF, GATA, SP1) produce qualitatively similar decay profiles, but the absolute sensitivity varies with motif length and composition.
- gNIAH complements the perturbation-variance ECL by providing a motif-specific, biologically grounded view of context utilization.

For real models, replace `SyntheticModel` with a genomic model wrapper and use realistic sequence lengths (e.g., 196,608 bp for Enformer).